# Demo 00 - setup and reset

Run the whole notebook top to bottom, both the first time and again right before the demo. It
builds the governed home for the demo data (own schema, own volumes) and puts everything back to
zero, so the ingestion counters start empty in front of the audience.

In [0]:
dbutils.widgets.text("login", "")
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container", "")

login           = dbutils.widgets.get("login")
catalog         = dbutils.widgets.get("target_catalog")
storage_account = dbutils.widgets.get("storage_account")
container       = dbutils.widgets.get("container")

assert all([login, catalog, storage_account, container])

demo    = f"{login}_demo_bronze"       # where the demo lands
lab     = f"{login}_bronze"            # lab schema, only read from
abfss   = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

landing = f"/Volumes/{catalog}/{demo}/demo_landing"
chk     = f"/Volumes/{catalog}/{demo}/checkpoints"

print(f"{catalog}.{demo}")

## Governed home

Schema with a managed location inside our own container, one external volume for files arriving
from outside, one managed volume for stream state. Both go through the Unity Catalog external
location on the shared storage credential, so no access key appears anywhere.

In [0]:
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalog}.{demo}
              MANAGED LOCATION '{abfss}/demo'""")

spark.sql(f"""CREATE EXTERNAL VOLUME IF NOT EXISTS {catalog}.{demo}.demo_landing
              LOCATION '{abfss}/demo_landing'""")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{demo}.checkpoints")

display(spark.sql(f"SHOW VOLUMES IN {catalog}.{demo}"))

## Reset

Three things have to go together, and forgetting one of them is how a demo produces duplicates or
silently loads nothing:

- the **tables**, otherwise old rows survive the reload
- the **checkpoints**, otherwise the streams think they have already read everything
- the **generated files** in the landing zone, otherwise the third device is already there before
  the demo starts

The registry file is left alone on purpose. It is the one file that is supposed to be sitting there
when the demo begins.

In [0]:
spark.sql(f"DROP VIEW IF EXISTS {catalog}.{demo}.v_readings_flagged")

for t in ["device_registry_bronze", "sensor_readings_bronze"]:
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.{demo}.{t}")
    print("dropped table", t)

# names must match the checkpoint paths in demo_01, demo_03 and demo_05
for sub in ["registry", "readings", "backfill"]:
    try:
        dbutils.fs.rm(f"{chk}/{sub}", True)
        print("cleared checkpoint", sub)
    except Exception:
        print("checkpoint", sub, "was not there")

# the third device's backfill files, written by demo_04
try:
    dbutils.fs.rm(f"{landing}/readings", True)
    print("cleared generated backfill files")
except Exception:
    print("no backfill files to clear")

## Seed the landing zone

The device registry is the batch source: a file exported from another system and dropped into the
landing zone every now and then. Here it comes from the Lab 2 landing volume, so nothing has to be
uploaded by hand before the demo.

The copy is skipped if the file is already there, so running this twice does not leave a second
copy that Auto Loader would count as new data.

In [0]:
src_registry = f"/Volumes/{catalog}/{lab}/landing/camera_dim.csv"
dst_registry = f"{landing}/registry/camera_dim.csv"


def exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


if exists(dst_registry):
    print("registry already in place, leaving it alone")
else:
    dbutils.fs.cp(src_registry, dst_registry)
    print("copied", src_registry, "->", dst_registry)

display(dbutils.fs.ls(f"{landing}/registry"))

In [0]:
# what the demo reads from, and how much of it there is
for t in ["qnap_stats_bronze", "raw_events_bronze", "camera_dim_bronze"]:
    print(f"{t:22} {spark.table(f'{catalog}.{lab}.{t}').count():>9,} rows")